# 03 - Modello M1: LSTM Multi-Step Diretto
### Deep Learning for Forecasting Quantum Dynamics – PXP Many-Body Model

<a href="https://colab.research.google.com/github/Riccardo-27/Quantum-Dynamics/blob/main/notebooks/03_model_m1_lstm_direct.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

### 🎯 Obiettivi del Notebook:
1. **Implementazione di M1 (LSTM Diretto Multi-Step)**: Costruzione della prima architettura neurale della nostra progressione di ricerca.
2. **Paradigma Single-Shot**: L'encoder LSTM riassume l'intera storia $X \in \mathbb{R}^{B \times T_{in} \times 55}$ in un vettore di contesto $h_T \in \mathbb{R}^{B \times H}$, mentre una testa densa MLP mappa direttamente sull'intero orizzonte target $Y \in \mathbb{R}^{B \times T_{out} \times 55}$.
3. **Training Engine Ottimizzato**: Supporto trasparente per GPU (CUDA / Google Colab T4) e CPU locale, con AdamW, gradient clipping, learning rate scheduler ed Early Stopping.
4. **Analisi della Sfida 1 (Time Window Sensitivity)**: Confronto sistematico delle prestazioni tra:
   - $\text{Short Window}$: $T_{in} = 50$ ($1.0\text{ s}$)
   - $\text{Medium Window}$: $T_{in} = 100$ ($2.0\text{ s}$)
   - $\text{Long Window}$: $T_{in} = 200$ ($4.0\text{ s}$)
   su orizzonte comune $T_{out} = 100$ ($2.0\text{ s}$).
5. **Diagnostica e Logging Centralizzato**:
   - Curve di perdita (Loss MSE).
   - Rollout predetto vs traiettoria quantistica reale ($\ge 300\text{ DPI}$).
   - Curva di decadimento dell'errore lungo l'orizzonte futuro $\text{MSE}(h)$.
   - Registrazione automatica dei risultati in `results/results_log.csv`.
6. **Dichiarazione del Limite di M1**: Motivazione teorica e quantitativa che giustifica l'introduzione del modello successivo **M2 (LSTM Seq2Seq Autoregressivo)**.


In [ ]:
import os
import sys
from pathlib import Path

# Rilevamento automatico ambiente: Google Colab vs Locale
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Ambiente rilevato: GOOGLE COLAB (Cloud Execution)")
    from google.colab import drive
    drive.mount("/content/drive")
    
    REPO_NAME = "Quantum-Dynamics"
    REPO_PATH = Path(f"/content/{REPO_NAME}")
    
    if not REPO_PATH.exists():
        !git clone https://github.com/Riccardo-27/Quantum-Dynamics.git {REPO_PATH}
    else:
        !cd {REPO_PATH} && git pull
        
    os.chdir(REPO_PATH)
    PROJECT_ROOT = REPO_PATH
else:
    PROJECT_ROOT = Path(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

print(f"Directory di lavoro: {PROJECT_ROOT}")


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from utils.seeding import set_seeds
from utils.data_loading import load_dataset
from utils.windowing import get_dataloaders, WINDOW_CONFIGS
from utils.results_logger import log_experiment_result
from utils.metrics import compute_metrics, compute_horizon_mse
from utils.viz import plot_predictions, plot_loss_curves
from models.lstm_direct import LSTMDirectForecaster

# Fissaggio rigoroso della riproducibilità
SEED = 42
set_seeds(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device di calcolo: {device}")
if device.type == "cuda":
    print(f"GPU attiva: {torch.cuda.get_device_name(0)}")
    print(f"Memoria VRAM disponibile: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 1. Caricamento Dati e Indici Ufficiali


In [ ]:
data_dir = PROJECT_ROOT / "data"

# Caricamento del dataset ad alta velocità
features, time_grid = load_dataset(str(data_dir))

# Caricamento degli indici ufficiali congelati nello Step 1
split_data = np.load(data_dir / "split_indices.npz")
split_indices = {
    "train_idx": split_data["train_idx"],
    "val_idx": split_data["val_idx"],
    "test_idx": split_data["test_idx"],
}

print(f"Dataset pronto: {features.shape} | Traiettorie: 320 Train, 40 Val, 40 Test")


## 2. Formulazione del Modello M1 (LSTM Diretto)

### Equazioni e Funzionamento:
Dato l'input passato $\mathbf{X} = [\mathbf{s}_1, \dots, \mathbf{s}_{T_{in}}] \in \mathbb{R}^{B \times T_{in} \times 55}$:
1. **Rappresentazione Sequenziale**:
   $$(\mathbf{h}_t, \mathbf{c}_t) = \text{LSTM}(\mathbf{s}_t, \mathbf{h}_{t-1}, \mathbf{c}_{t-1}), \quad t = 1, \dots, T_{in}$$
2. **Vettore di Contesto**:
   $$\mathbf{z} = \text{LayerNorm}(\mathbf{h}_{T_{in}})$$
3. **Proiezione Multi-Step Diretta**:
   $$\hat{\mathbf{Y}}_{\text{flat}} = \mathbf{W}_2 \cdot \text{GELU}(\mathbf{W}_1 \mathbf{z} + \mathbf{b}_1) + \mathbf{b}_2 \in \mathbb{R}^{B \times (T_{out} \cdot 55)}$$
4. **Reshape Finale**:
   $$\hat{\mathbf{Y}} = \text{Reshape}(\hat{\mathbf{Y}}_{\text{flat}}) \in \mathbb{R}^{B \times T_{out} \times 55}$$

### Vantaggi:
- Velocissimo da addestrare in parallelo.
- Nessuna divergenza o instabilità dovuta ad autoregressione durante il training.

### Limite Dichiarato (che Motiva M2):
- Non modella esplicitamente la causalità temporale interna tra i singoli timestep futuri: $\hat{\mathbf{s}}_{t+1}$ e $\hat{\mathbf{s}}_{t+2}$ vengono predetti contemporaneamente senza mutuo scambio di informazione.


In [ ]:
# Istanziazione di test per verificare i parametri
sample_model = LSTMDirectForecaster(
    input_features=55,
    output_features=55,
    input_len=50,
    output_len=100,
    hidden_dim=128,
    num_layers=2,
    dropout=0.1
)

n_params = sample_model.count_parameters()
print(f"Modello M1 istanziato con successo!")
print(f"Parametri totali addestrabili: {n_params:,}")


## 3. Training & Validation Engine

Definiamo funzioni di addestramento pulite con:
- Ottimizzatore `AdamW` con weight decay per regolarizzazione.
- Gradient Clipping ($||\mathbf{g}|| \le 1.0$) per evitare esplosioni di gradiente nelle RNN.
- `ReduceLROnPlateau` per dimezzare il learning rate se la validation loss smette di migliorare.
- Early Stopping rigoroso per evitare overfitting.


In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device, clip_grad=1.0):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        y_pred = model(x_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()

        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)

        optimizer.step()
        total_loss += loss.item() * len(x_batch)

    return total_loss / len(dataloader.dataset)


def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item() * len(x_batch)
    return total_loss / len(dataloader.dataset)


def fit_model(
    model,
    train_loader,
    val_loader,
    epochs=25,
    lr=1e-3,
    weight_decay=1e-4,
    patience=7,
    checkpoint_path=None,
    device=device,
):
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    criterion = nn.MSELoss()

    train_losses = []
    val_losses = []
    best_val_loss = float("inf")
    patience_counter = 0

    print(f"--- Inizio Addestramento ({epochs} epoche max, device: {device}) ---")
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        t_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss = evaluate(model, val_loader, criterion, device)

        train_losses.append(t_loss)
        val_losses.append(v_loss)
        scheduler.step(v_loss)

        improved = ""
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            patience_counter = 0
            improved = " [* BEST CHECKPOINT]"
            if checkpoint_path:
                os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
                torch.save(model.state_dict(), checkpoint_path)
        else:
            patience_counter += 1

        print(f"Epoca {epoch:02d}/{epochs:02d} | Train MSE: {t_loss:.6f} | Val MSE: {v_loss:.6f}{improved}")

        if patience_counter >= patience:
            print(f"Early stopping attivato all'epoca {epoch}!")
            break

    total_time = time.time() - start_time
    print(f"Addestramento completato in {total_time:.2f} s | Miglior Val MSE: {best_val_loss:.6f}")

    # Ricarica il miglior modello
    if checkpoint_path and os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "best_val_loss": best_val_loss,
        "total_time": total_time,
    }


## 4. Esperimento 1: Addestramento M1 su `short_window` ($T_{in} = 50$)

Addestriamo il Modello M1 sulla configurazione più sfidante: finestra di input di soli $50$ timestep ($1.0\text{ s}$) per prevedere i successivi $100$ timestep ($2.0\text{ s}$).


In [ ]:
# 1. Caricamento DataLoader per short_window
train_loader_short, val_loader_short, test_loader_short, cfg_short = get_dataloaders(
    features=features,
    split_indices=split_indices,
    config_name="short_window",
    stride_train=5,
    stride_eval=10,
    batch_size=64
)

# 2. Inizializzazione Modello M1
model_m1_short = LSTMDirectForecaster(
    input_features=55,
    output_features=55,
    input_len=cfg_short["input_len"],
    output_len=cfg_short["output_len"],
    hidden_dim=128,
    num_layers=2,
    dropout=0.1
)

ckpt_short = str(PROJECT_ROOT / "results" / "checkpoints" / "m1_lstm_direct_short.pt")

# 3. Addestramento
history_short = fit_model(
    model=model_m1_short,
    train_loader=train_loader_short,
    val_loader=val_loader_short,
    epochs=20,
    lr=1e-3,
    patience=5,
    checkpoint_path=ckpt_short,
    device=device
)


## 5. Valutazione e Metriche sul Test Set (`short_window`)


In [ ]:
# Inferenza su tutto il Test Set
model_m1_short.eval()
test_preds = []
test_targets = []

with torch.no_grad():
    for x_b, y_b in test_loader_short:
        x_b = x_b.to(device)
        pred = model_m1_short(x_b).cpu().numpy()
        test_preds.append(pred)
        test_targets.append(y_b.numpy())

test_preds = np.concatenate(test_preds, axis=0)
test_targets = np.concatenate(test_targets, axis=0)

metrics_short = compute_metrics(test_targets, test_preds)
print("=== RISULTATI TEST SET (M1 - Short Window) ===")
print(f"Test MSE:  {metrics_short['mse']:.6f}")
print(f"Test MAE:  {metrics_short['mae']:.6f}")
print(f"Test RMSE: {metrics_short['rmse']:.6f}")

# Registrazione automatica su results_log.csv
log_experiment_result({
    "run_id": "M1_LSTM_Direct_short_window",
    "model_name": "M1_LSTM_Direct",
    "family": "RNN",
    "window_name": "short_window",
    "input_len": 50,
    "output_len": 100,
    "epochs": len(history_short["train_losses"]),
    "learning_rate": 1e-3,
    "batch_size": 64,
    "train_mse": history_short["train_losses"][-1],
    "val_mse": history_short["best_val_loss"],
    "test_mse": metrics_short["mse"],
    "test_mae": metrics_short["mae"],
    "num_parameters": model_m1_short.count_parameters(),
    "notes": "Single-shot projection baseline, no autoregression",
}, log_path=str(PROJECT_ROOT / "results" / "results_log.csv"))


## 6. Visualizzazioni Diagnostiche

### 6.1 Curve di Apprendimento
Confrontiamo la discesa della perdita durante le epoche.


In [ ]:
fig_dir = PROJECT_ROOT / "results" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

plot_loss_curves(
    train_losses=history_short["train_losses"],
    val_losses=history_short["val_losses"],
    title="M1 (LSTM Diretto) - Curve di Loss su Short Window",
    save_path=str(fig_dir / "m1_loss_curves_short.png"),
    dpi=300
)


### 6.2 Traiettoria Reale vs Predetta su Campione di Test


In [ ]:
# Selezioniamo il primo campione del test set per visualizzare magnetizzazioni e correlazioni
sample_true = test_targets[0]
sample_pred = test_preds[0]

time_horizon = time_grid[50:150]  # Orizzonte temporale di previsione target

plot_predictions(
    y_true=sample_true,
    y_pred=sample_pred,
    feature_indices=[0, 1, 2, 9, 10, 20],
    feature_names=["m_1 (Qubit 1)", "m_2 (Qubit 2)", "m_3 (Qubit 3)", "m_{10} (Qubit 10)", "c_{1,2} (Coppia 1-2)", "c_{2,4} (Coppia 2-4)"],
    time_steps=time_horizon,
    title="M1 (LSTM Diretto): Dinamica Quantistica Reale vs Predetta (Test Set)",
    save_path=str(fig_dir / "m1_prediction_sample.png"),
    dpi=300
)


### 6.3 Decadimento dell'Accuratezza lungo l'Orizzonte Futuro ($\text{MSE}(h)$)

Calcoliamo l'errore quadratico medio per ciascuno dei $100$ passi futuri per quantificare come decade la qualità della predizione all'aumentare della distanza temporale dal presente.


In [ ]:
horizon_mse = compute_horizon_mse(test_targets, test_preds)

plt.figure(figsize=(10, 4.5))
plt.plot(range(1, 101), horizon_mse, color="#1f77b4", linewidth=2.0, label="M1: MSE(h)")
plt.title("M1: Profilo di Errore lungo l'Orizzonte di Previsione Futuro ($h = 1 \dots 100$)", fontsize=13)
plt.xlabel("Passi nel Futuro $h$ (ogni passo = 0.02 s)", fontsize=11)
plt.ylabel("MSE per Timestep", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend()

plt.savefig(fig_dir / "m1_horizon_mse_drift.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Analisi della Sfida 1: Sensibilità alla Finestra di Input

Valutiamo ora come cambiano le prestazioni di M1 aumentando la finestra di osservazione iniziale:
- **Short Window** ($T_{in} = 50$, già addestrato)
- **Medium Window** ($T_{in} = 100$)
- **Long Window** ($T_{in} = 200$)


In [ ]:
# Addestramento rapido su medium_window
train_loader_med, val_loader_med, test_loader_med, cfg_med = get_dataloaders(
    features=features,
    split_indices=split_indices,
    config_name="medium_window",
    stride_train=5,
    stride_eval=10,
    batch_size=64
)

model_m1_med = LSTMDirectForecaster(
    input_features=55,
    output_features=55,
    input_len=cfg_med["input_len"],
    output_len=cfg_med["output_len"],
    hidden_dim=128,
    num_layers=2,
    dropout=0.1
)

ckpt_med = str(PROJECT_ROOT / "results" / "checkpoints" / "m1_lstm_direct_medium.pt")
hist_med = fit_model(
    model=model_m1_med,
    train_loader=train_loader_med,
    val_loader=val_loader_med,
    epochs=15,
    lr=1e-3,
    patience=4,
    checkpoint_path=ckpt_med,
    device=device
)

# Test evaluation medium
model_m1_med.eval()
t_preds_med, t_targs_med = [], []
with torch.no_grad():
    for x_b, y_b in test_loader_med:
        t_preds_med.append(model_m1_med(x_b.to(device)).cpu().numpy())
        t_targs_med.append(y_b.numpy())
m_med = compute_metrics(np.concatenate(t_targs_med), np.concatenate(t_preds_med))

log_experiment_result({
    "run_id": "M1_LSTM_Direct_medium_window",
    "model_name": "M1_LSTM_Direct",
    "family": "RNN",
    "window_name": "medium_window",
    "input_len": 100,
    "output_len": 100,
    "epochs": len(hist_med["train_losses"]),
    "learning_rate": 1e-3,
    "batch_size": 64,
    "train_mse": hist_med["train_losses"][-1],
    "val_mse": hist_med["best_val_loss"],
    "test_mse": m_med["mse"],
    "test_mae": m_med["mae"],
    "num_parameters": model_m1_med.count_parameters(),
    "notes": "Sensitivity test medium window",
}, log_path=str(PROJECT_ROOT / "results" / "results_log.csv"))

print(f"Confronto Sfida 1:")
print(f"  - Short Window (T_in=50):   Test MSE = {metrics_short['mse']:.6f}")
print(f"  - Medium Window (T_in=100): Test MSE = {m_med['mse']:.6f}")


## 8. Conclusioni e Motivazione Scientifica per il Modello M2

### Risultati Chiave di M1:
- Abbiamo stabilito la **prima solida baseline neurale** per la predizione della dinamica del modello PXP a 10 qubit.
- Il modello mappa rapidamente $T_{in} \to T_{out}$ senza problemi di stabilità ricorsiva.
- L'analisi della **Sfida 1** dimostra quantitativamente come l'estensione della finestra di osservazione riduca l'errore di previsione.

---

### ⚠️ Il Limite Teorico di M1: Perché serve il Modello M2?
Nonostante i buoni risultati della baseline:
1. **Assenza di Causalità Futura**: M1 produce i 100 timestep futuri attraverso una singola proiezione densa parallela. Non esiste alcun meccanismo che imponga che $\hat{\mathbf{s}}_{t+2}$ sia fisicamente condizionato da $\hat{\mathbf{s}}_{t+1}$.
2. **Orizzonte Rigido**: Il modello può predire solo un numero fisso di passi pari a $T_{out} = 100$; non può effettuare rollout ricorsivi indefiniti.

👉 **Nel prossimo notebook (`04_model_m2_lstm_seq2seq.ipynb`)**, risolveremo esattamente questo limite introducendo **M2: LSTM Encoder-Decoder Autoregressivo**, con decodifica passo-passo, **Scheduled Sampling** e valutazione pura in modalità **Free-Running**!
